# 예제 5: Claude API로 설비 이상 보고서 자동 생성

---

## 이 예제에서 배울 것
- **LLM API 활용**: Python 코드로 Claude에게 질문하고 답변 받기
- **프롬프트 엔지니어링**: AI에게 원하는 결과를 얻기 위한 지시문 작성법
- **업무 자동화**: 데이터 → 자동 분석 → 자동 보고서 생성 파이프라인 구축

## 핵심 메시지
> **"데이터를 AI에게 보여주면, 보고서가 자동으로 완성된다."**  
> 코딩 실력보다 **'어떤 것을 분석할지'와 '어떻게 질문할지'**가 더 중요합니다.

## 제조 현장 연관
- MES/SCADA 센서 데이터 → 자동 이상 보고서 생성
- 주간 품질 보고서, 설비 현황 요약 자동화
- **리더가 할 일**: 보고서 검토 + 의사결정 (작성은 AI가)

---
## API 키 안내
이 예제를 완전히 실행하려면 **Claude API 키**가 필요합니다.  
- 강사가 교육용 API 키를 제공하거나
- 개인 키: [console.anthropic.com](https://console.anthropic.com) 에서 발급
- **API 키 없이도** → 예시 출력이 자동으로 표시됩니다

In [1]:
# ─────────────────────────────────────────────────────────
# anthropic 라이브러리 설치
# Claude API를 파이썬에서 사용하기 위한 공식 라이브러리
# ─────────────────────────────────────────────────────────
!pip install -q anthropic
print("✅ anthropic 라이브러리 설치 완료!")

✅ anthropic 라이브러리 설치 완료!


In [2]:
# ─────────────────────────────────────────────────────────
# 필요한 도구 불러오기
# ─────────────────────────────────────────────────────────
import anthropic  # Claude API 클라이언트
import json       # JSON 형식 데이터 처리 (딕셔너리 ↔ 텍스트 변환)

In [12]:
# 2단계: API 키 안전하게 입력하기
# getpass를 사용하면 입력한 키가 화면에 표시되지 않습니다
# ⚠️ 절대 API 키를 코드에 직접 입력하지 마세요!
import getpass

API_KEY = getpass.getpass("🔑 Anthropic API 키를 입력하세요: ")
print("✅ API 키가 설정되었습니다 (보안을 위해 표시되지 않습니다)")

✅ API 키가 설정되었습니다 (보안을 위해 표시되지 않습니다)


## STEP 1: 설비 센서 데이터 준비

실제 현장에서는 MES/SCADA 시스템에서 이 데이터를 자동으로 가져옵니다.  
이 예제에서는 딕셔너리(dict) 형태로 직접 입력합니다.

In [4]:
# ─────────────────────────────────────────────────────────
# 설비 센서 데이터 정의
# 딕셔너리(dict): {키: 값} 형태로 여러 정보를 묶어서 저장
#   → 엑셀에서 하나의 행(row)에 여러 열(column)이 있는 것과 유사
# ─────────────────────────────────────────────────────────
설비_데이터 = {
    # 기본 정보
    "설비명":    "압출기 #3 (전선 외피 압출)",
    "측정일시":  "2026-04-19 14:30",
    "담당자":    "김보전 주임",

    # 센서 측정값 (중첩 딕셔너리: 딕셔너리 안에 또 딕셔너리)
    "센서_데이터": {
        "진동": {
            "현재": 2.8,
            "정상범위": "0.5 ~ 1.5 mm/s",
            "경고수준": "2.0 이상",
            "상태": "⚠️ 경고"  # 정상 범위 초과
        },
        "온도": {
            "현재": 89,
            "정상범위": "70 ~ 85°C",
            "경고수준": "85 이상",
            "상태": "⚠️ 경고"
        },
        "전류": {
            "현재": 18.2,
            "정상범위": "14 ~ 17 A",
            "경고수준": "17 이상",
            "상태": "🔴 위험"  # 경고 수준보다 높음
        },
        "소음": {
            "현재": 78,
            "정상범위": "60 ~ 75 dB",
            "경고수준": "75 이상",
            "상태": "⚠️ 경고"
        }
    },

    # 최근 고장 이력 (리스트: 여러 항목을 순서대로 저장)
    "최근_고장_이력": [
        {"일자": "2025-12-10", "내용": "베어링 교체 (내경 마모)"},
        {"일자": "2025-08-22", "내용": "모터 과열로 2시간 비계획 정지"}
    ],

    "현재_생산_현황": "420개/시간 (목표: 500개/시간, 달성률: 84%)"
}

print("✅ 설비 데이터 준비 완료!")
print("\n 현재 센서 상태 요약:")

# 센서 상태 출력
for 센서이름, 정보 in 설비_데이터['센서_데이터'].items():
    # .items(): 딕셔너리의 키-값 쌍을 순서대로 꺼내기
    print(f"  {센서이름:4s}: {정보['현재']} (정상: {정보['정상범위']}) → {정보['상태']}")

✅ 설비 데이터 준비 완료!

 현재 센서 상태 요약:
  진동  : 2.8 (정상: 0.5 ~ 1.5 mm/s) → ⚠️ 경고
  온도  : 89 (정상: 70 ~ 85°C) → ⚠️ 경고
  전류  : 18.2 (정상: 14 ~ 17 A) → 🔴 위험
  소음  : 78 (정상: 60 ~ 75 dB) → ⚠️ 경고


## STEP 2: 프롬프트 작성 — AI에게 어떻게 질문할 것인가?

**좋은 프롬프트의 3가지 요소:**
1. **역할 부여**: "당신은 XX 전문가입니다"
2. **맥락 제공**: 분석할 데이터 또는 상황
3. **출력 형식 지정**: 어떤 형태로 답변해 달라고

In [6]:
# ─────────────────────────────────────────────────────────
# 프롬프트(Prompt) 작성
#
# f-string(f"""..."""):  
#   f 앞에 붙이면 중괄호 {} 안에 변수를 넣을 수 있습니다
#   예: f"안녕 {이름}님" → "안녕 김철수님"
#
# json.dumps(딕셔너리, ensure_ascii=False, indent=2):
#   딕셔너리를 보기 좋은 JSON 텍스트로 변환
#   ensure_ascii=False: 한글이 깨지지 않도록
#   indent=2: 들여쓰기 2칸으로 보기 좋게
# ─────────────────────────────────────────────────────────
프롬프트 = f"""당신은 15년 경력의 전선 제조 설비 보전 전문가입니다.
아래 설비 모니터링 데이터를 분석하고, 현장 관리자가 즉시 읽고 행동할 수 있는
설비 이상 보고서를 작성해주세요.

=== 설비 센서 데이터 ===
{json.dumps(설비_데이터, ensure_ascii=False, indent=2)}
========================

아래 형식으로 보고서를 작성해주세요:

## 설비 이상 보고서

### 1. 종합 상태 판정
[정상/주의/경보/긴급 중 하나 + 한 줄 요약]

### 2. 이상 항목 상세 분석
[각 이상 센서의 원인 추정과 상호 연관성 분석]

### 3. 권장 조치 (긴급도 순)
[즉시/오늘 내/이번 주 내 조치를 구분하여 체크리스트 형태로]

### 4. 방치 시 예상 리스크
[시간대별 구체적 피해 시나리오와 예상 비용]

### 5. 중장기 예방 권고
[재발 방지를 위한 제안]

간결하고 전문적으로, 현장에서 즉시 실행 가능한 수준으로 작성해주세요.
"""

print(" 프롬프트 작성 완료!")
print(f"프롬프트 길이: {len(프롬프트)} 글자")
print("\n 프롬프트 미리보기 (처음 200자):")
print(프롬프트[:200] + "...")

 프롬프트 작성 완료!
프롬프트 길이: 1184 글자

 프롬프트 미리보기 (처음 200자):
당신은 15년 경력의 전선 제조 설비 보전 전문가입니다.
아래 설비 모니터링 데이터를 분석하고, 현장 관리자가 즉시 읽고 행동할 수 있는
설비 이상 보고서를 작성해주세요.

=== 설비 센서 데이터 ===
{
  "설비명": "압출기 #3 (전선 외피 압출)",
  "측정일시": "2026-04-19 14:30",
  "담당자": "김보전 주임",
  "센...


## STEP 3: Claude API 호출 — 보고서 자동 생성

In [13]:
print(len(API_KEY))

108


In [14]:
# ─────────────────────────────────────────────────────────
# Claude API 호출하여 보고서 생성
# ─────────────────────────────────────────────────────────

# API 키 유효성 확인
API_유효 = API_KEY != "YOUR_API_KEY_HERE" and len(API_KEY) > 20

if API_유효:
    print("Claude API 호출 중... (약 5~15초 소요)")
    print("=" * 60)

    try:
        # anthropic.Anthropic(): Claude API 클라이언트 생성
        # api_key: 내 API 키 (인증)
        client = anthropic.Anthropic(api_key=API_KEY)

        # client.messages.create(): Claude에게 메시지 전송
        response = client.messages.create(
            model="claude-opus-4-5",  # 사용할 Claude 모델
            # claude-haiku-xxx: 빠르고 저렴 (간단한 작업)
            # claude-sonnet-xxx: 균형적 (대부분의 작업)
            # claude-opus-xxx:  느리지만 가장 정확 (복잡한 분석)

            max_tokens=1500,  # 최대 응답 길이 (토큰 수)
            # 1토큰 ≈ 한글 0.5글자, 영어 0.75단어

            messages=[
                {
                    "role": "user",      # 역할: user(질문자) 또는 assistant(AI)
                    "content": 프롬프트  # 실제 질문 내용
                }
            ]
        )

        # 응답 텍스트 추출
        # response.content: 응답 내용 목록
        # [0].text: 첫 번째 응답의 텍스트
        보고서 = response.content[0].text
        print(보고서)

        print("\n" + "=" * 60)
        print(f"✅ 보고서 생성 완료! ({response.usage.output_tokens} 토큰 사용)")

    except Exception as e:
        print(f"❌ API 오류 발생: {e}")
        print("   → API 키를 확인하거나 예시 출력을 참고하세요.")
        API_유효 = False

if not API_유효:
    print("ℹ️  API 키 없이 실행 중 — 아래는 예상 출력 예시입니다:")
    print("   (실제 API 사용 시 이보다 더 상세한 내용이 생성됩니다)")
    print("=" * 60)
    예시_보고서 = """
## 설비 이상 보고서

### 1. 종합 상태 판정
**🔴 긴급** — 전류·온도·진동 3개 항목 동시 이상, 즉각적 조치 필요

### 2. 이상 항목 상세 분석
**전류 18.2A** (정상: 14~17A, 초과: +7.1%):
  - 2025-08-22 모터 과열 이력 고려 시 모터 권선 열화 가능성 높음
  - 전류 증가 → 열 발생 증가 → 온도 상승의 악순환 진행 중

**온도 89°C** (정상: 70~85°C, 초과: +4°C):
  - 전류 증가에 따른 모터 발열이 주원인
  - 현재 추세 지속 시 30분 내 90°C 초과 예상

**진동 2.8mm/s** (정상: 0.5~1.5, 초과: +87%):
  - 2025-12-10 베어링 교체 후에도 재발 → 미스얼라인먼트 또는 잦은 교체 환경 원인 의심

### 3. 권장 조치 (긴급도 순)
**즉시 (지금 당장):**
- [ ] 압출기 #3 부하 20~30% 감소 (속도 줄이기)
- [ ] 보전팀 긴급 호출
- [ ] 10분 간격 수동 모니터링 시작

**오늘 내:**
- [ ] 모터 절연 저항 측정 (Megger 테스트)
- [ ] 베어링 온도 접촉 측정 및 이상음 확인
- [ ] 냉각팬 작동 여부 확인

**이번 주 내:**
- [ ] 모터 전문 점검 의뢰 (과열 이력 2회 → 교체 검토)
- [ ] 얼라인먼트 측정 및 조정

### 4. 방치 시 예상 리스크
- **2시간 내**: 모터 과부하 보호 장치 작동 → 비계획 정지
- **피해 추정**: 생산 손실 약 1,680개 + 긴급 수리비 500~800만원
- **최악의 경우**: 모터 소손 → 교체 비용 1,500만원 + 5일 정지

### 5. 중장기 예방 권고
- 모터 전류 모니터링 주기: 1시간 → 15분으로 단축
- 베어링 교체 주기 재검토 (6개월 → 4개월 단축 권고)
- 예지보전 AI 시스템 도입 검토 (본 예제가 바로 그 시스템입니다!)
"""
    print(예시_보고서)

Claude API 호출 중... (약 5~15초 소요)
# 설비 이상 보고서

| 설비명 | 압출기 #3 (전선 외피 압출) |
|--------|---------------------------|
| **측정일시** | 2026-04-19 14:30 |
| **담당자** | 김보전 주임 |
| **작성자** | 설비보전팀 |

---

## 1. 종합 상태 판정

### 🔴 **긴급 (CRITICAL)**

> **베어링 열화 및 모터 과부하 복합 이상 — 4개 센서 전수 경고/위험 상태로 6시간 내 비계획 정지 가능성 높음**

| 항목 | 현재값 | 정상범위 | 초과율 | 상태 |
|------|--------|----------|--------|------|
| 전류 | 18.2A | 14~17A | **+7.1%** | 🔴 위험 |
| 진동 | 2.8mm/s | 0.5~1.5mm/s | **+87%** | ⚠️ 경고 |
| 온도 | 89°C | 70~85°C | **+4.7%** | ⚠️ 경고 |
| 소음 | 78dB | 60~75dB | **+4%** | ⚠️ 경고 |

---

## 2. 이상 항목 상세 분석

### 📊 원인 추정 및 연관성 분석

```
[근본 원인 추정]
            ┌─────────────────┐
            │  베어링 재열화   │ ← 2025-12 교체 후 4개월, 조기 마모 의심
            │  (주축 또는 감속기)│
            └────────┬────────┘
                     │
        ┌────────────┼────────────┐
        ▼            ▼            ▼
   ┌─────────┐  ┌─────────┐  ┌─────────┐
   │ 진동↑   │→│ 마찰↑   │→│ 발열↑   │
   │ 2.8mm/s │  │ 소음78dB│  │ 89°C    │
   └─────────┘  └────────

## STEP 4: 활용 심화 — 주간 품질 보고서 자동 생성

In [8]:
# ─────────────────────────────────────────────────────────
# 주간 품질 현황 보고서 자동 생성 예시
# (API 키 없어도 프롬프트 구조 학습 가능)
# ─────────────────────────────────────────────────────────

# 주간 분석 결과 데이터 (실제는 데이터 분석 후 자동 생성)
품질_분석결과 = {
    "기간": "2026년 4월 3주차 (04.13~04.19)",
    "총_생산량": 12450,
    "총_불량수": 187,
    "평균_불량률": "1.50%",
    "전주_불량률": "1.23%",
    "변화": "+0.27%p (악화)",
    "라인별": {
        "라인A": "1.12%",
        "라인B": "0.98%",
        "라인C": "2.40% ← 특이 증가"
    },
    "주요_불량_유형": [
        {"유형": "절연 두께 불균일", "건수": 89, "비율": "47.6%"},
        {"유형": "외관 스크래치",   "건수": 56, "비율": "29.9%"},
        {"유형": "길이 오차",       "건수": 42, "비율": "22.5%"}
    ]
}

# 주간 보고서용 프롬프트
품질_보고서_프롬프트 = f"""당신은 전선 제조 공장의 품질 관리 전문가입니다.
아래 주간 품질 분석 결과를 바탕으로, 생산 관리자에게 보고할
주간 품질 현황 보고서를 작성해주세요.

=== 분석 결과 ===
{json.dumps(품질_분석결과, ensure_ascii=False, indent=2)}
=================

보고서 형식:
1. 이번 주 핵심 요약 (3줄 이내)
2. 주요 이슈 및 원인 추정
3. 이번 주 긴급 조치 사항 (담당자 | 내용 | 기한)
4. 다음 주 모니터링 포인트

간결하게, 수치 중심으로, A4 반 장 분량으로 작성해주세요.
"""

print("📋 주간 품질 보고서 프롬프트 작성 완료!")
print("\n이 프롬프트를 ChatGPT나 Claude 웹사이트에 직접 붙여넣어도 됩니다:")
print("-" * 60)
print(품질_보고서_프롬프트[:500] + "...")
print("-" * 60)
print("\n💡 핵심 포인트:")
print("  1. Python으로 데이터를 계산 (숫자/통계 작업 → 파이썬이 잘함)")
print("  2. 계산 결과를 LLM에게 전달 (문장/보고서 작업 → LLM이 잘함)")
print("  3. LLM이 전문가 수준의 보고서를 자동 생성")
print("\n  → 파이썬 + LLM의 조합이 최강 업무 자동화 파이프라인!")

📋 주간 품질 보고서 프롬프트 작성 완료!

이 프롬프트를 ChatGPT나 Claude 웹사이트에 직접 붙여넣어도 됩니다:
------------------------------------------------------------
당신은 전선 제조 공장의 품질 관리 전문가입니다.
아래 주간 품질 분석 결과를 바탕으로, 생산 관리자에게 보고할
주간 품질 현황 보고서를 작성해주세요.

=== 분석 결과 ===
{
  "기간": "2026년 4월 3주차 (04.13~04.19)",
  "총_생산량": 12450,
  "총_불량수": 187,
  "평균_불량률": "1.50%",
  "전주_불량률": "1.23%",
  "변화": "+0.27%p (악화)",
  "라인별": {
    "라인A": "1.12%",
    "라인B": "0.98%",
    "라인C": "2.40% ← 특이 증가"
  },
  "주요_불량_유형": [
    {
      "유형": "절연 두께 불균일",
      "건수": 89,
      "비율": "47.6%"
    },
    {
      "유형": "외관 스크래치",
      "건수": 56,
      "비율": "29.9%"
    },
    {
      "유형...
------------------------------------------------------------

💡 핵심 포인트:
  1. Python으로 데이터를 계산 (숫자/통계 작업 → 파이썬이 잘함)
  2. 계산 결과를 LLM에게 전달 (문장/보고서 작업 → LLM이 잘함)
  3. LLM이 전문가 수준의 보고서를 자동 생성

  → 파이썬 + LLM의 조합이 최강 업무 자동화 파이프라인!


## STEP 5: 나만의 보고서 자동화 템플릿 만들기

이 셀을 수정하여 **내 업무에 맞는 프롬프트**를 만들어보세요!

In [9]:
# ─────────────────────────────────────────────────────────
# 📝 내 업무 자동화 프롬프트 템플릿
# 아래 변수들을 내 상황에 맞게 수정해보세요!
# ─────────────────────────────────────────────────────────

# ── 수정 가능한 설정 ─────────────────────────────────────
AI_역할 = "전선 제조 공장의 생산 관리 전문가"  # ← 내 업무에 맞게 변경

내_데이터 = """
  [여기에 분석할 데이터나 상황을 입력하세요]
  예시:
  - 이번 주 생산량: 12,450개
  - 불량률: 1.5% (목표: 1.2%)
  - 라인C 불량률 급증: 2.4%
"""

원하는_결과 = """생산 관리자에게 보고할 주간 현황 요약"""

출력_형식 = """
  1. 핵심 요약 (3줄 이내)
  2. 주요 이슈와 원인
  3. 조치 사항 (담당자 | 내용 | 기한)
"""
# ─────────────────────────────────────────────────────────

# 내 프롬프트 생성
내_프롬프트 = f"""당신은 {AI_역할}입니다.
아래 데이터를 분석하여 {원하는_결과}를 작성해주세요.

데이터:
{내_데이터}

작성 형식:
{출력_형식}
"""

print("✅ 내 프롬프트 생성 완료!")
print("\n📋 생성된 프롬프트:")
print("-" * 50)
print(내_프롬프트)
print("-" * 50)
print("\n💡 이 프롬프트를 ChatGPT/Claude에 붙여넣어 사용하세요!")
print("   또는 위의 API 코드에서 '프롬프트' 변수를 '내_프롬프트'로 바꿔서 실행")

✅ 내 프롬프트 생성 완료!

📋 생성된 프롬프트:
--------------------------------------------------
당신은 전선 제조 공장의 생산 관리 전문가입니다.
아래 데이터를 분석하여 생산 관리자에게 보고할 주간 현황 요약를 작성해주세요.

데이터:

  [여기에 분석할 데이터나 상황을 입력하세요]
  예시:
  - 이번 주 생산량: 12,450개
  - 불량률: 1.5% (목표: 1.2%)
  - 라인C 불량률 급증: 2.4%


작성 형식:

  1. 핵심 요약 (3줄 이내)
  2. 주요 이슈와 원인
  3. 조치 사항 (담당자 | 내용 | 기한)


--------------------------------------------------

💡 이 프롬프트를 ChatGPT/Claude에 붙여넣어 사용하세요!
   또는 위의 API 코드에서 '프롬프트' 변수를 '내_프롬프트'로 바꿔서 실행


## 🎓 정리 및 핵심 포인트

### 배운 것
1. **LLM API 호출 구조**: client 생성 → messages.create() → 응답 텍스트 추출
2. **좋은 프롬프트**: 역할 부여 + 맥락(데이터) 제공 + 출력 형식 지정
3. **파이썬 + LLM 조합**: 데이터 처리는 Python, 문서 작성은 LLM
4. **API 모델 선택**: 간단한 작업은 Haiku, 복잡한 분석은 Sonnet/Opus

### 업무 자동화 파이프라인 설계
```
데이터 수집 (MES/ERP) 
      ↓
Python으로 계산/분석 (예제 1~4)
      ↓
결과를 LLM에 전달 (이번 예제)
      ↓
자동 보고서 생성 → 이메일/메신저 전송
```

### 활용 아이디어
- 일일 생산 현황 보고서 자동 생성 (매일 오전 8시)
- 설비 이상 감지 즉시 → 문자/카카오톡 자동 발송
- 고객 클레임 데이터 분석 → 원인 추정 보고서 자동화

### 다음 단계
**예제 6**에서는 지금까지 배운 모든 기술을 합쳐서 **스마트팩토리 대시보드**를 만듭니다! 📊

---
*송현그룹 mAX Academy 3기 | 예제 5/6*